[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

*Attention & Transformers*
Implement attention the **FlashAttention** way: stream over key/value tiles,
never materializing the full $T_q \times T_k$ score matrix, and produce output
that is *mathematically exact* — the same function as standard attention, not an
approximation of it.

### Signature
```python
def flash_attention(q, k, v, block_size=16):
    # q: (T_q, d), k: (T_k, d), v: (T_k, d_v)
    ...  # -> (T_q, d_v)
```

### The online softmax recurrence
For each key block, with scores $s$ and current state $(m, \ell, \text{acc})$:

$$m^{\text{new}} = \max(m, \max s) \qquad
\alpha = e^{\,m - m^{\text{new}}}$$

$$\ell^{\text{new}} = \alpha\ell + \sum e^{\,s - m^{\text{new}}} \qquad
\text{acc}^{\text{new}} = \alpha\,\text{acc} + e^{\,s - m^{\text{new}}}V_{\text{block}}$$

Start at $m = -\infty$, $\ell = 0$, $\text{acc} = 0$; finish with
$\text{acc}/\ell$.

### Rules
- Process keys in tiles of `block_size`; never build the full score matrix
- Scale by $1/\sqrt{d}$
- Must agree with standard attention to floating-point tolerance, for **every**
  `block_size` — an approximation that is merely close is a fail
- Handle a `T_k` that is not a multiple of `block_size`
- Do not use `jax.nn.softmax` on the whole matrix

### Why the rescaling is what makes tiling *exact*
Softmax needs a global max for stability, but a streaming algorithm has not seen
the future when it processes block 1. The fix is to keep the max *so far*, and
when a later block raises it, retroactively correct everything already
accumulated by $e^{m_{\text{old}} - m_{\text{new}}}$.

Because $e^{s-m_{\text{old}}} \cdot e^{m_{\text{old}}-m_{\text{new}}} =
e^{s-m_{\text{new}}}$ exactly, the correction is not an approximation — it is an
algebraic identity. In exact arithmetic Flash and naive attention compute the
same number; in float32 they differ only by the rounding of a different
summation order (expect $\sim10^{-7}$, and the demo shows it). That is a
completely different kind of "different" from [[linear_attention]], which drops
the softmax and changes the answer at any precision. Be precise about this in an
interview: FlashAttention is **exact, not bit-identical**.

### It is an IO win, not a FLOP win
FlashAttention does the **same** number of floating-point operations as standard
attention — slightly more, in fact, because of the rescaling. It is faster
because attention is **memory-bandwidth bound**: the naive version writes an
$O(T^2)$ score matrix out to HBM and reads it back for the softmax and again for
the $V$ multiply. Flash keeps each tile in SRAM and never writes the scores at
all, taking HBM traffic from $\Theta(T^2 + Td)$ words down to
$\Theta(T^2 d^2 / M)$, where $M$ is the SRAM capacity in words (Dao et al.,
2022, Thm. 2). Since $M \gg d^2$ on real accelerators, that is a large
constant-factor cut — it is still quadratic in $T$.

The memory consequence is the bigger deal in practice: peak activation memory
for attention drops from $O(T^2)$ to $O(T)$, which is what made long context
affordable at all. Note that this Python/XLA version demonstrates the
*algorithm* — the actual speedup requires a fused kernel (Pallas/Triton/CUDA)
that controls SRAM residency directly.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def flash_attention(q, k, v, block_size=16):
    """Tiled attention with an online softmax.

    Args:
        q:          (T_q, d)
        k:          (T_k, d)
        v:          (T_k, d_v)
        block_size: number of keys processed per tile

    Returns:
        (T_q, d_v) — the same function as standard attention, for any block_size.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

q = jax.random.normal(jax.random.key(0), (8, 16))
k = jax.random.normal(jax.random.key(1), (40, 16))
v = jax.random.normal(jax.random.key(2), (40, 4))

ref = jax.nn.softmax(q @ k.T / jnp.sqrt(16.0), axis=-1) @ v

for bs in (4, 7, 16, 64):
    out = flash_attention(q, k, v, block_size=bs)
    print(f"block_size={bs:>3}: max |diff| vs standard = {float(jnp.abs(out - ref).max()):.2e}")
# ~1e-7 for every tiling: that is float32 round-off from a different summation
# order, not approximation error. The rescaling is an algebraic identity.

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("flash_attention")

# hint("flash_attention")      # stuck? nudge without the answer
# solution("flash_attention")  # spoiler: the reference implementation